# 085 — Espacios latentes y autoencoders variacionales

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución explicada

**Ejercicio 1.** KL(σ_q=0.8) = −log 0.8 + (0.64 + 0.25)/2 − 0.5 = 0.2231 + 0.4450 − 0.5
= **0.1681**. KL(σ_q=1.2) = −log 1.2 + (1.44 + 0.25)/2 − 0.5 = −0.1823 + 0.8450 − 0.5
= **0.1627**. Sorpresa razonable: son casi iguales — la KL penaliza tanto encoger
(σ<1) como ensanchar (σ>1) la posterior respecto al prior; no es una distancia en σ.

**Ejercicio 2.** z ∈ {−0.3, 0.5, 0.9, 1.3}. Con infinitos ε, z ~ N(0.5, 0.8²): media
0.5 y desviación 0.8, porque la transformación afín conserva la gaussianidad. Como
z = μ + σ·ε es determinista dado ε, ∂z/∂μ = 1 y ∂z/∂σ = ε existen y el gradiente
retropropaga; el muestreo "crudo" no define esas derivadas.

**Ejercicio 3.** Con ε = 0.5: z = 0.9, g = 1.8, log p(x|z) = −0.9189 − 0.02 = −0.9389,
ELBO = −0.9389 − 0.1681 = **−1.1071**. Con ε = −0.5: z = 0.1, g = 0.2,
log p(x|z) = −0.9189 − 1.62 = −2.5389, ELBO = **−2.7071**. Difieren porque el ELBO es
una esperanza sobre q y cada ε da un estimado de una muestra; entrenar promedia ese
ruido (estimador insesgado de baja varianza gracias a la reparametrización).

**Ejercicio 4.** El contrato JSON expone `kind` y `evidence`; solo la evidencia
autoriza conclusiones sobre lo que el laboratorio demuestra.

In [ ]:
result_a = run_lab("generation", seed=85)
result_b = run_lab("generation", seed=850)
assert result_a["kind"] == "generation" and result_a["evidence"]
print("claves:", sorted(result_a.keys()))
print("misma estructura:", sorted(result_a.keys()) == sorted(result_b.keys()))


In [ ]:
# Verificación numérica de los ejercicios
import math

def kl_1d(mu_q, sigma_q, mu_p=0.0, sigma_p=1.0):
    return (math.log(sigma_p / sigma_q)
            + (sigma_q**2 + (mu_q - mu_p)**2) / (2 * sigma_p**2) - 0.5)

# Ejercicio 1
kl_a = kl_1d(0.5, 0.8)
kl_b = kl_1d(0.5, 1.2)
print(f"KL(sigma=0.8) = {kl_a:.4f}   KL(sigma=1.2) = {kl_b:.4f}")
assert abs(kl_a - 0.1681) < 1e-3 and abs(kl_b - 0.1627) < 1e-3

# Ejercicio 2
mu, sigma = 0.5, 0.8
for eps in (-1.0, 0.0, 0.5, 1.0):
    print(f"eps={eps:+.1f}  z = {mu + sigma * eps:+.2f}")

# Ejercicio 3
x = 2.0
def elbo(eps):
    z = mu + sigma * eps
    g = 2 * z
    log_px_z = -0.5 * math.log(2 * math.pi) - (x - g) ** 2 / 2
    return log_px_z - kl_a
for eps in (0.5, -0.5):
    print(f"eps={eps:+.1f}  ELBO(1 muestra) = {elbo(eps):.4f}")
assert abs(elbo(0.5) - (-1.1071)) < 1e-3
assert abs(elbo(-0.5) - (-2.7071)) < 1e-3

## Reflexión

1. Si el término KL de todas las dimensiones latentes cae a ≈0 durante el entrenamiento, ¿qué le pasó al modelo (posterior collapse) y por qué las reconstrucciones pueden seguir siendo buenas?
2. ¿Por qué sin el truco de reparametrización no fluye el gradiente del término de reconstrucción hacia los parámetros φ del encoder, y por qué REINFORCE sería una alternativa peor?
3. ¿Qué diferencia hay entre interpolar en el espacio latente de un VAE y en el de un autoencoder determinista, y qué término del ELBO explica esa diferencia?